scVI SOLO to be run on the DD and Scrublet completed datasets.
using MPS on mac

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/h5ad_files/inprogress_dd_scr.h5ad')

In [ ]:
adata

# Working SOLO code.

In [ ]:
# --- SOLO (scvi-tools) with automatic MPS/CUDA/CPU selection; no DataSplitter needed ---
import torch, numpy as np, random
import scvi
import pandas as pd

# Reproducibility
torch.manual_seed(0); np.random.seed(0); random.seed(0)

# ---------- Device selection ----------
def choose_accelerator_and_devices():
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return "mps", 1
    if torch.cuda.is_available():
        return "gpu", 1
    return "cpu", 1

accelerator, devices = choose_accelerator_and_devices()
try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass

print(f"Using accelerator={accelerator}, devices={devices}")

# Sanity checks
assert "counts" in adata.layers, "Put raw counts in adata.layers['counts']"
assert "batch"  in adata.obs,    "Put sample IDs in adata.obs['batch']"

# ---------- scVI setup ----------
scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="batch")

scvi_model = scvi.model.SCVI(
    adata,
    n_layers=2,
    n_latent=30,
    gene_likelihood="nb",
)

BATCH_SIZE = 2048  # lower if you see OOM on MPS

# Train scVI (no datamodule; pass batch_size directly)
scvi_model.train(
    max_epochs=100,
    check_val_every_n_epoch=10,
    early_stopping=True,
    accelerator=accelerator,
    devices=devices,
    batch_size=BATCH_SIZE,
    precision="32-true",        # AMP not supported on MPS
    plan_kwargs={"lr": 1e-3},
)

# ---------- SOLO on top of trained scVI ----------
from scvi.external import SOLO
solo = SOLO.from_scvi_model(scvi_model)

solo.train(
    max_epochs=100,
    early_stopping=True,
    accelerator=accelerator,
    devices=devices,
    batch_size=BATCH_SIZE,
    precision="32-true",
)

# ---------- Predict & store (robust to different scvi-tools versions) ----------
pred_df = solo.predict(adata)  # DataFrame indexed by obs_names

cols = [c.lower() for c in pred_df.columns]

if set(cols) >= {"doublet", "singlet"}:
    # Two-probability output
    # score = P(doublet); call = argmax (doublet vs singlet)
    score = pred_df.loc[:, [c for c in pred_df.columns if c.lower() == "doublet"][0]].astype(float)
    call  = pred_df.loc[:, [c for c in pred_df.columns if c.lower() in ("doublet", "singlet")]] \
                 .astype(float).idxmax(axis=1).str.lower().eq("doublet")

    score_col_used = [c for c in pred_df.columns if c.lower() == "doublet"][0]
    pred_col_used  = "argmax(doublet,singlet)"

elif any("prob" in c or "score" in c for c in cols):
    # Has a probability/score column already (keep first match)
    score_col_used = next(c for c in pred_df.columns if ("prob" in c.lower() or "score" in c.lower()))
    score = pred_df[score_col_used].astype(float)

    # Try to find a predicted label column; else fallback to threshold 0.5
    pred_cols = [c for c in pred_df.columns if ("pred" in c.lower() or "doublet" in c.lower())]
    if pred_cols:
        pred_col_used = pred_cols[0]
        if pred_df[pred_col_used].dtype == "object":
            call = pred_df[pred_col_used].astype(str).str.contains("doublet", case=False)
        else:
            # sometimes it's already boolean
            call = pred_df[pred_col_used].astype(bool)
    else:
        pred_col_used = "score>=0.5"
        call = score >= 0.5

elif any(c == "doublet" for c in cols):
    # Single "doublet" column (often the probability)
    score_col_used = [c for c in pred_df.columns if c.lower() == "doublet"][0]
    score = pred_df[score_col_used].astype(float)
    pred_col_used = "doublet>=0.5"
    call = score >= 0.5

else:
    raise RuntimeError(f"Unrecognized SOLO predict() columns: {pred_df.columns.tolist()}")

adata.obs["solo_score"] = score
adata.obs["solo_is_doublet"] = call.astype(bool)

adata.uns["solo_info"] = {
    **adata.uns.get("solo_info", {}),
    "score_col": score_col_used,
    "pred_col": pred_col_used,
}
print(
    "SOLO done:",
    int(adata.obs["solo_is_doublet"].sum()),
    "doublets flagged. Using score:", adata.uns["solo_info"]["score_col"],
    "and call:", adata.uns["solo_info"]["pred_col"]
)


In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(
    adata,
    color = ['solo_is_doublet'],
    frameon = False,
    size = 10
)

# Plotting the Model Training Metrics

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# See what was logged
print("scVI history keys:", list(scvi_model.history.keys()))

# Helper to safely get a series
def _get_hist(name):
    h = scvi_model.history.get(name, None)
    if h is None:
        return None
    # scvi logs as list of dicts or list of floats depending on version
    if isinstance(h, (list, tuple)):
        # if list of dicts with "step" or "epoch"
        if len(h) > 0 and isinstance(h[0], dict):
            return pd.DataFrame(h).set_index(pd.RangeIndex(len(h)))
        else:
            return pd.Series(h)
    if isinstance(h, pd.DataFrame) or isinstance(h, pd.Series):
        return h
    # fallback
    try:
        return pd.Series(h)
    except Exception:
        return None

# Common keys across versions (some may not exist depending on scvi-tools version)
train_keys = ["elbo_train", "reconstruction_loss_train", "kl_local_train"]
val_keys   = ["elbo_validation", "reconstruction_loss_validation", "kl_local_validation"]

# Plot ELBO (train/val)
plt.figure()
elbo_tr = _get_hist("elbo_train")
elbo_va = _get_hist("elbo_validation")
if elbo_tr is not None:
    (elbo_tr if isinstance(elbo_tr, pd.Series) else elbo_tr.squeeze()).plot(label="ELBO train")
if elbo_va is not None:
    (elbo_va if isinstance(elbo_va, pd.Series) else elbo_va.squeeze()).plot(label="ELBO val")
plt.title("scVI ELBO over epochs")
plt.xlabel("epoch")
plt.ylabel("ELBO (higher is better)")
plt.legend()
plt.show()

# Optionally plot reconstruction and KL terms (if present)
for tr_key, va_key, title in [
    ("reconstruction_loss_train", "reconstruction_loss_validation", "Reconstruction loss (lower is better)"),
    ("kl_local_train", "kl_local_validation", "KL local (regularization)"),
]:
    tr = _get_hist(tr_key); va = _get_hist(va_key)
    if tr is None and va is None:
        continue
    plt.figure()
    if tr is not None:
        (tr if isinstance(tr, pd.Series) else tr.squeeze()).plot(label="train")
    if va is not None:
        (va if isinstance(va, pd.Series) else va.squeeze()).plot(label="val")
    plt.title(f"scVI {title}")
    plt.xlabel("epoch"); plt.legend(); plt.show()

In [ ]:
adata

In [ ]:
adata.write_h5ad(f'{ACC_DATA_ROOT}/h5ad_files/inprogress_dd_scrub_solo.h5ad', compression='lzf')